# Gen AI Evaluation Service

Agent Platform's Gen AI evaluation service empowers you to assess any generative AI model or application according to your specific requirements.

Instead of relying solely on general leaderboards and reports, you can define your own evaluation criteria and directly compare how different models perform against your unique needs, use case and data.

This service allows you to:

- Gain a deeper understanding of model performance. And, Go beyond general metrics and understand how a model handles your specific data and tasks.
- Make informed decisions throughout the development lifecycle. By using evaluations to guide model selection, refine prompt engineering, and optimize model customization.
- Streamline your evaluation workflow by leveraging Agent Platform's integrated tools to easily launch and reuse evaluations as needed.

Essentially, it puts you in control of evaluating generative AI, ensuring the models you choose are the best fit for your specific applications.

## Use Case: Recipe Assistant

Throughout this notebook we will evaluate a single application: a **Recipe Assistant** built on Gemini. The assistant has two jobs:

1. **Summarize recipes** so that users can quickly understand how a dish is made.
2. **Search for recipes** by calling a `search_recipes` tool (function calling) based on a user's request.

We will use the same set of recipes for every evaluation technique so that you can focus on *how* each technique works and *when* to use it.

## Learning Objectives

In this notebook, you will learn:

- How to use Gen AI Evaluation Service
- Different types of evaluation techniques: computation-based, model-based (pointwise and pairwise) and tool use quality metrics
- How to prepare your dataset and get it ready for evaluation
- Analyze and understand the results from evaluation

### Setup

In [1]:
import json
import pandas as pd
import vertexai
from google import genai
from google.genai import types
from vertexai import types as eval_types

LOCATION = "global"

# Initialize Vertex AI Client (for evaluation) and Google GenAI Client (for direct generation / function calling)
eval_client = vertexai.Client(location=LOCATION)
client = genai.Client(vertexai=True, location=LOCATION)

## Evaluation Process

Agent Platform's Gen AI evaluation service lets you assess any generative model based on your specific needs and criteria.

These are the four steps you will follow to help you evaluate:

1. Define: Tailor metrics to your business goals and choose your evaluation approach (Computation based vs Model Based).
2. Prepare: Create a dataset that reflects your real-world use case.
3. Run: Generate responses with `eval_client.evals.run_inference()` (or supply your own responses) and score them with `eval_client.evals.evaluate()`.
4. Analyze: Interpret your results using `.show()`, `summary_metrics`, and `win_rates` to understand how each model performs against your specific criteria.

Let's take a look at these steps one by one

### Recipe Dataset

This is the shared dataset for the whole notebook. `context` holds three recipe articles that the Recipe Assistant will summarize, and `reference` holds a human-written "ground truth" summary for each one. Different evaluation techniques will use different parts of this dataset:

| Technique | Uses `context` | Uses `reference` |
| --- | --- | --- |
| Computation-based metrics | Yes | Yes (required) |
| Model-based pointwise metrics | Yes | No |
| Model-based pairwise metrics | Yes | No |

**Exercise** Create a `eval_dataset` DataFrame with the columns `context`, `reference` and `instruction`

In [ ]:
instruction = "Summarize the following recipe"

context = [
    "To make a classic spaghetti carbonara, start by bringing a large pot of salted water to a boil. While the water is heating up, cook pancetta or guanciale in a skillet with olive oil over medium heat until it's crispy and golden brown. Once the pancetta is done, remove it from the skillet and set it aside. In the same skillet, whisk together eggs, grated Parmesan cheese, and black pepper to make the sauce. When the pasta is cooked al dente, drain it and immediately toss it in the skillet with the egg mixture, adding a splash of the pasta cooking water to create a creamy sauce.",
    "Preparing a perfect risotto requires patience and attention to detail. Begin by heating butter in a large, heavy-bottomed pot over medium heat. Add finely chopped onions and minced garlic to the pot, and cook until they're soft and translucent, about 5 minutes. Next, add Arborio rice to the pot and cook, stirring constantly, until the grains are coated with the butter and begin to toast slightly. Pour in a splash of white wine and cook until it's absorbed. From there, gradually add hot chicken or vegetable broth to the rice, stirring frequently, until the risotto is creamy and the rice is tender with a slight bite.",
    "To bake a decadent chocolate cake from scratch, start by preheating your oven to 350°F (175°C) and greasing and flouring two 9-inch round cake pans. In a large mixing bowl, cream together softened butter and granulated sugar until light and fluffy. Beat in eggs one at a time, making sure each egg is fully incorporated before adding the next. In a separate bowl, sift together all-purpose flour, cocoa powder, baking powder, baking soda, and salt. Divide the batter evenly between the prepared cake pans and bake for 25-30 minutes, or until a toothpick inserted into the center comes out clean.",
]

reference = [
    "The process of making spaghetti carbonara involves boiling pasta, crisping pancetta or guanciale, whisking together eggs and Parmesan cheese, and tossing everything together to create a creamy sauce.",
    "Preparing risotto entails sautéing onions and garlic, toasting Arborio rice, adding wine and broth gradually, and stirring until creamy and tender.",
    "Baking a decadent chocolate cake requires creaming butter and sugar, beating in eggs, combining the dry ingredients and baking the batter until a toothpick comes out clean.",
]

eval_dataset = None  # TODO create a pandas dataframe here

eval_dataset

We will use the Gen AI Evaluation SDK on `vertexai.Client` (`eval_client.evals`). For full details, see the [Run evaluations with the Gen AI evaluation SDK](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/evaluation-genai-sdk) guide.

#### Understand the `evals` workflow

With `eval_client.evals`, evaluation follows a simple two-step workflow (`run_inference()` and `evaluate()`):

- **Bring-your-own-response (BYOR)**: If you already generated responses (e.g., summaries or tool-call outputs), pass a `pandas.DataFrame` containing `prompt` and `response` columns (plus optional `reference`) directly to `eval_client.evals.evaluate(dataset=..., metrics=...)`.
- **Run inference and evaluate**: Pass a `pandas.DataFrame` with a `prompt` column (and optional `reference`) to `eval_client.evals.run_inference(model=..., src=...)` to generate responses into an `EvaluationDataset`, then pass that dataset to `eval_client.evals.evaluate(dataset=..., metrics=...)`.
- **Compare multiple models side-by-side**: Pass a list of `EvaluationDataset` objects to `eval_client.evals.evaluate(dataset=[ds_a, ds_b], metrics=...)` to score multiple models on the same prompts and compute win rates automatically.

In this notebook you will see all three modes in action.

## 1. Computation Based Metrics

[Computation based metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#computation-based-metrics) are computed using mathematical formulas to compare the model's output against a **ground truth or reference**. Commonly used computation-based metrics include ROUGE and BLEU. The commonly used metrics can be categorized into the following groups:

- Lexicon-based metrics: Use math to calculate the string similarities between LLM-generated results and ground truth, such as `Exact Match` and `ROUGE`.
- Count-based metrics: Aggregate the number of rows that hit or miss certain ground-truth labels, such as `F1-score`, `Accuracy`, and `Tool Name Match`.
- Embedding-based metrics: Calculate the distance between the LLM-generated results and ground truth in the embedding space, reflecting their level of similarity. These include `cosine similarity` and `euclidean distance`

For our Recipe Assistant, let's use computation based metrics to decide **which prompt template produces the best recipe summaries**. Because these metrics need a ground truth, we will use both the `context` and `reference` columns of our recipe dataset.

Now, we'll define different prompt templates to evaluate. This list can include all the different prompt templates you want to experiment with. For the purpose of demostration, we'll use two different prompt templates below.

In [ ]:
prompt_templates = [
    "Instruction: {instruction}. Article: {context}. Summary:",
    "Article: {context}. Complete this task: {instruction}, in one sentence. Summary:",
]

Next, we will define the metrics to measure using `vertexai.types`. Computation-based metrics such as `rouge_1`, `rouge_l_sum`, and `bleu` are specified via `eval_types.Metric(name=...)`, and managed safety evaluation is specified via `eval_types.RubricMetric.SAFETY`. See the [Define your evaluation metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval) documentation for the full list of supported metrics.

In [ ]:
metrics = [
    eval_types.Metric(name="rouge_1"),
    eval_types.Metric(name="rouge_l_sum"),
    eval_types.Metric(name="bleu"),
    eval_types.RubricMetric.SAFETY,
]

Now that we have configured all the parameters, we are ready to evaluate.

**Exercise** For each prompt template, format the `prompt` column alongside `reference` in `prompt_df`, call `eval_client.evals.run_inference(model="gemini-4-hydrogen", src=prompt_df)`, and then pass the resulting `EvaluationDataset` and `metrics` to `eval_client.evals.evaluate()`. See the [Gen AI evaluation SDK documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/evaluation-genai-sdk).

In [ ]:
eval_results = {}

for i, prompt_template in enumerate(prompt_templates):
    prompt_df = pd.DataFrame(
        {
            "prompt": [
                prompt_template.format(
                    instruction=row.instruction, context=row.context
                )
                for row in eval_dataset.itertuples()
            ],
            "reference": eval_dataset["reference"],
        }
    )

    inference_dataset = None  # TODO: Call eval_client.evals.run_inference(model="gemini-4-hydrogen", src=prompt_df)
    eval_results[f"Prompt #{i}"] = None  # TODO: Call eval_client.evals.evaluate(dataset=inference_dataset, metrics=metrics)

In [ ]:
for prompt_name, eval_result in eval_results.items():
    print(f"=== {prompt_name} ===")
    eval_result.show()

## 2. Model Based Metrics

Computation based metrics require a reference, and they only measure word overlap, not whether a summary is actually *good*. When you do not have ground truths, or want to measure qualitative properties, you can evaluate your model using another LLM. This technique is called **Model based evaluation**. Model-based metrics assesses your candidate model against a judge model. The judge model for most use cases is Gemini, but you can also use models such as [MetricX](https://github.com/google-research/metricx) or [COMET](https://huggingface.co/Unbabel/wmt22-comet-da) for translation use cases.

You can measure model-based metrics pairwise or pointwise:

- **Pointwise metrics**: Let the judge model assess the candidate model's output based on the evaluation criteria. For example, the score could be 0 to 5, where 0 means the response does not fit the criteria, while 5 means the response fits the criteria well.

- **Pairwise metrics**: Let the judge model compare the responses of two models and pick the better one. This is often used when comparing a candidate model with the baseline model. Pairwise metrics are only supported with Gemini as a judge model.

### 2a. Pointwise Metrics

Now that we have picked a prompt template, we want to rate the recipe summaries on two qualitative criteria that matter for a Recipe Assistant: **fluency** and **actionability** (can a home cook follow it?). We will define a custom `LLMMetric` called `text_quality` using `eval_types.MetricPromptBuilder`.

In [ ]:
# Custom definition of text_quality using LLMMetric and MetricPromptBuilder.
text_quality = eval_types.LLMMetric(
    name="text_quality",
    prompt_template=eval_types.MetricPromptBuilder(
        criteria={
            "fluency": "Sentences flow smoothly and are easy to read, avoiding awkward phrasing or run-on sentences. Ideas and sentences connect logically, using transitions effectively where needed.",
            "actionability": "The summary captures the key ingredients and cooking steps in a logical order so that a home cook could understand how to prepare the dish.",
        },
        rating_scores={
            "2": "The response performs well on both criteria.",
            "1": "The response is somewhat aligned with both criteria",
            "0": "The response falls short on both criteria",
        },
        evaluation_steps={
            "Step 1": "Assess the response in aspects of all criteria provided. Provide assessment according to each criterion.",
            "Step 2": 'Score based on the Rating Scores and explain your reasoning. Output ONLY valid JSON with keys "explanation" (string) and "score" (number).',
        },
    ),
)

Let's take a look at what this `text_quality` metric looks like.

In [ ]:
print(text_quality.prompt_template)

This time we will use the **bring-your-own-response (BYOR)** mode. We first generate summaries of the same recipes with our model of interest (`gemini-4-hydrogen`) and store both `prompt` and `response` columns in a DataFrame. Notice that no `reference` is needed.

**Exercise:** For the `context` list from the Recipe Dataset, generate the summary responses from Gemini (`gemini-4-hydrogen`) using `client.models.generate_content()`.

In [ ]:
prompt_template = "Instruction: {instruction}. Article: {context}. Summary:"

prompts = []
responses = []
for article in context:
    prompt = None  # TODO: Format prompt_template with instruction and article
    response = None  # TODO: Call client.models.generate_content(model="gemini-4-hydrogen", contents=prompt).text
    prompts.append(prompt)
    responses.append(response)
responses

**Exercise**: Define a dataset for evaluation `pointwise_eval_dataset` as a pandas DataFrame with the `prompt` and `response` columns generated above.

In [ ]:
pointwise_eval_dataset = None  # TODO: Define a DataFrame with "prompt" and "response" columns

In [ ]:
pointwise_eval_results = eval_client.evals.evaluate(
    dataset=pointwise_eval_dataset,
    metrics=[text_quality],
)

You can view the `summary_metrics` aggregated over all 3 recipes here.

In [ ]:
pointwise_eval_results.summary_metrics

To inspect the per-recipe scores and judge explanations for `text_quality`, render the interactive evaluation report with `.show()`.

In [ ]:
pointwise_eval_results.show()

In the interactive report above (or via `pointwise_eval_results.eval_case_results`), you can read the rationale behind the judge model's score for each recipe summary.

### 2b. Compare Models Side-by-Side (SxS)

Next, we want to decide **which model should power our Recipe Assistant**: `gemini-4-argon` or `gemini-4-hydrogen`.

With the Gen AI Evaluation SDK, you run inference for each model separately using `eval_client.evals.run_inference()`, and then pass both resulting `EvaluationDataset` objects as a list to `eval_client.evals.evaluate()`. The service evaluates each candidate against the same rubric and automatically computes win and tie rates.

In [ ]:
sxs_prompts_df = pd.DataFrame(
    {
        "prompt": [
            prompt_template.format(
                instruction=row.instruction, context=row.context
            )
            for row in eval_dataset.itertuples()
        ]
    }
)

Once we have prepared the prompt dataset, we generate inference responses for the two models we want to compare: `gemini-4-argon` and `gemini-4-hydrogen`.

In [ ]:
argon_eval_dataset = eval_client.evals.run_inference(
    model="gemini-4-argon",
    src=sxs_prompts_df,
)
hydrogen_eval_dataset = eval_client.evals.run_inference(
    model="gemini-4-hydrogen",
    src=sxs_prompts_df,
)

Rather than writing a pairwise judge prompt from scratch, Agent Platform provides managed adaptive rubric metrics in `eval_types.RubricMetric` (documented in [Define your evaluation metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval)). Here, we use `eval_types.RubricMetric.TEXT_QUALITY`, which dynamically generates linguistic quality rubrics (assessing fluency, coherence, and grammar) and evaluates both models side-by-side.

Once we have generated inference results for both models, we can run the side-by-side evaluation.

**Exercise** Call `eval_client.evals.evaluate()` passing `[argon_eval_dataset, hydrogen_eval_dataset]` for `dataset` and `[eval_types.RubricMetric.TEXT_QUALITY]` for `metrics`.

In [ ]:
sxs_eval_result = None  # TODO: Call eval_client.evals.evaluate(dataset=[...], metrics=[...])

Once the evaluation finishes, you can view the side-by-side comparison report with `.show()` and inspect `win_rates`.

In [ ]:
print("Win rates:", sxs_eval_result.win_rates)
sxs_eval_result.show()

## 3. Tool Use Quality Metrics

Besides summarizing recipes, our Recipe Assistant also needs to find recipes for users. It does this by interacting with an external recipe database through **Function Calling**.

The Gen AI Evaluation SDK provides specialized computation-based metrics to assess tool use accuracy:
- `tool_call_valid`: Verifies that the model produced a syntactically valid function call.
- `tool_name_match`: Checks whether the model selected the correct tool/function name.
- `tool_parameter_key_match`: Evaluates whether all expected parameter keys were included.
- `tool_parameter_kv_match`: Evaluates whether both parameter keys and their corresponding values match the ground-truth reference.

Suppose our assistant has access to a tool named `search_recipes` that searches a recipe repository using parameters:
- `recipe_name` (string): Title of the dish (e.g., "Spaghetti Carbonara", "Wild Mushroom Risotto").
- `cuisine` (string): Cuisine style or tradition (e.g., "Italian").
- `servings` (string): Desired number of servings.
- `dietary_restriction` (string): Dietary preferences (e.g., "gluten-free", "vegetarian").
- `max_prep_time` (string): Preparation time limit (e.g., "30 minutes").
- `difficulty` (string): Cooking difficulty (e.g., "easy", "medium", "hard").

In [ ]:
tool_metrics = [
    eval_types.Metric(name="tool_call_valid"),
    eval_types.Metric(name="tool_name_match"),
    eval_types.Metric(name="tool_parameter_key_match"),
    eval_types.Metric(name="tool_parameter_kv_match"),
]

### 3a. Evaluate a Bring-Your-Own-Response (BYOR) Dataset

You can evaluate tool use quality on pre-recorded model responses and expected ground-truth references.

Each tool call is formatted as a JSON string:
`{"content": "", "tool_calls": [{"name": "<function_name>", "arguments": {<key>: <value>}}]}`

Let's test four representative scenarios in our recipe assistant:
1. **Exact match**: Model correctly calls `search_recipes` for Spaghetti Carbonara with all parameters matching.
2. **Value mismatch**: Model calls `search_recipes` for Carbonara when the user requested Mushroom Risotto.
3. **Missing parameters (key mismatch)**: Model omits dietary restrictions and prep time.
4. **Value variations**: Model extracts different serving sizes and time constraints.

**Exercise** Create a `tool_eval_dataset` DataFrame with the columns `prompt`, `response`, and `reference`

In [ ]:
tool_prompts = [
    "Find a gluten-free Italian Spaghetti Carbonara recipe for 4 servings (max 30 minutes, medium difficulty).",
    "Find a vegetarian Italian Wild Mushroom Risotto recipe for 2 servings (max 45 minutes, medium difficulty).",
    "Find a gluten-free Italian Spaghetti Carbonara recipe for 4 servings (max 30 minutes, medium difficulty).",
    "Find a gluten-free Italian Spaghetti Carbonara recipe for 4 servings (max 30 minutes, medium difficulty).",
]

tool_responses = [
    # 1. Exact match (Spaghetti Carbonara)
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 2. Value mismatch (Carbonara instead of Risotto)
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 3. Missing parameters (only recipe_name and cuisine)
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                    },
                }
            ],
        }
    ),
    # 4. Multiple value differences (different servings, dietary restriction, prep time, difficulty)
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "6",
                        "dietary_restriction": "none",
                        "max_prep_time": "20 minutes",
                        "difficulty": "easy",
                    },
                }
            ],
        }
    ),
]

tool_references = [
    # 1. Expected: Exact match for Carbonara
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 2. Expected: Wild Mushroom Risotto
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Wild Mushroom Risotto",
                        "cuisine": "Italian",
                        "servings": "2",
                        "dietary_restriction": "vegetarian",
                        "max_prep_time": "45 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 3. Expected: Complete parameter set for Carbonara
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 4. Expected: Specific dietary and time constraints
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
]

tool_eval_dataset = None  # TODO: Create a pandas DataFrame with "prompt", "response", and "reference" columns

tool_eval_dataset

Evaluate tool usage quality across our test cases:

**Exercise** Call `eval_client.evals.evaluate()` with `dataset=tool_eval_dataset` and `metrics=tool_metrics`.

In [ ]:
tool_byor_result = None  # TODO: Call eval_client.evals.evaluate(dataset=tool_eval_dataset, metrics=tool_metrics)

tool_byor_result.show()

### 3b. Live Model Function Calling and Evaluation with Google GenAI SDK

Now let's test a live Gemini model (`gemini-4-hydrogen`) performing function calling using the Google GenAI SDK (`google.genai`), and evaluate its generated tool call against an expected reference.

We will:
1. Define the `search_recipes` tool declaration using `types.FunctionDeclaration` and `types.Tool`.
2. Send a natural language recipe search request to Gemini.
3. Extract and unpack the function call response into the format expected by the evaluation service.
4. Run `eval_client.evals.evaluate()` to evaluate the model's function calling precision.

In [ ]:
search_recipes_func = types.FunctionDeclaration(
    name="search_recipes",
    description="Search for recipes based on dish name, cuisine, servings, and dietary preferences.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "recipe_name": types.Schema(
                type=types.Type.STRING,
                description="The name of the recipe or dish to search for.",
            ),
            "cuisine": types.Schema(
                type=types.Type.STRING,
                description="The cuisine style or cuisine (e.g., Italian, French, American).",
            ),
            "servings": types.Schema(
                type=types.Type.STRING,
                description="The number of servings to prepare.",
            ),
            "dietary_restriction": types.Schema(
                type=types.Type.STRING,
                description="Dietary restrictions (e.g., gluten-free, vegetarian, dairy-free, none).",
            ),
            "max_prep_time": types.Schema(
                type=types.Type.STRING,
                description="Maximum preparation or cooking time (e.g., 30 minutes).",
            ),
            "difficulty": types.Schema(
                type=types.Type.STRING,
                description="Difficulty level (e.g., easy, medium, hard).",
            ),
        },
        required=[
            "recipe_name",
            "cuisine",
            "servings",
            "dietary_restriction",
            "max_prep_time",
            "difficulty",
        ],
    ),
)

search_recipes_tool = types.Tool(
    function_declarations=[search_recipes_func],
)

**Exercise:** Call `client.models.generate_content` with `gemini-4-hydrogen` and pass `search_recipes_tool` in `types.GenerateContentConfig` to generate a function call for the prompt below.

In [ ]:
prompt = """I'd like to find a recipe for classic Spaghetti Carbonara for 4 servings that is gluten-free, 
with a max prep time of 30 minutes and medium difficulty in the Italian cuisine tradition.
"""

gemini_response = None  # TODO: Call client.models.generate_content with model="gemini-4-hydrogen", contents=prompt, and tools=[search_recipes_tool]

# Inspect the function calls returned by Gemini
gemini_response.function_calls

In [ ]:
def unpack_response(response):
    """Extracts function calls from a GenerateContentResponse into the eval JSON format."""
    if response.function_calls:
        fc = response.function_calls[0]
        output = {"content": "", "tool_calls": [{"name": fc.name, "arguments": dict(fc.args)}]}
    else:
        fc = response.candidates[0].content.parts[0].function_call
        output = {"content": "", "tool_calls": [{"name": fc.name, "arguments": dict(fc.args)}]}
    return json.dumps(output)


response = unpack_response(gemini_response)
response

**Exercise**: Define a dataset for evaluation `live_eval_dataset` as a pandas DataFrame with the columns `prompt`, `response`, and `reference`.

In [ ]:
reference_str = json.dumps(
    {
        "content": "",
        "tool_calls": [
            {
                "name": "search_recipes",
                "arguments": {
                    "recipe_name": "Spaghetti Carbonara",
                    "cuisine": "Italian",
                    "servings": "4",
                    "dietary_restriction": "gluten-free",
                    "max_prep_time": "30 minutes",
                    "difficulty": "medium",
                },
            }
        ],
    }
)

live_eval_dataset = None  # TODO: Create a pandas DataFrame with "prompt", "response", and "reference" columns

In [ ]:
# Expected Tool Call Reference
json.loads(live_eval_dataset.reference[0])

In [ ]:
# Actual Gemini Tool Call Response
json.loads(live_eval_dataset.response[0])

**Exercise** Call `eval_client.evals.evaluate()` with `live_eval_dataset` and `tool_metrics` to evaluate the live model's tool call.

In [ ]:
live_eval_result = (
    None  # TODO: Call eval_client.evals.evaluate(dataset=live_eval_dataset, metrics=tool_metrics) here
)

live_eval_result.show()

To explore additional metrics, batch/BigQuery datasets, and agent evaluation workflows, see the [Gen AI Evaluation SDK documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/evaluation-genai-sdk).

## Summary and Key Takeaways

In this notebook, you used the **Gen AI Evaluation SDK (`vertexai.Client().evals`)** to evaluate a single **Recipe Assistant** application with every key evaluation technique:

| Technique | Question answered | Metrics | SDK workflow |
| --- | --- | --- | --- |
| Computation-based | Which prompt template writes the best recipe summaries? | `rouge_1`, `rouge_l_sum`, `bleu`, `RubricMetric.SAFETY` | `run_inference()` → `evaluate()` |
| Model-based pointwise | How fluent and actionable are the summaries? | Custom `text_quality` (`LLMMetric` + `MetricPromptBuilder`) | BYOR `evaluate()` |
| Side-by-side comparison | Should we use `gemini-4-argon` or `gemini-4-hydrogen`? | `RubricMetric.TEXT_QUALITY` | `run_inference()` per model → `evaluate([ds_a, ds_b])` |
| Tool use quality | Does the assistant call `search_recipes` correctly? | `tool_call_valid`, `tool_name_match`, `tool_parameter_key_match`, `tool_parameter_kv_match` | BYOR and live Gemini function calling `evaluate()` |

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.